# 02 Model Scientific Interest

Canonical modeling notebook for yearly scientific interest.
This notebook computes SI, ranks future high-interest pathogen-antibiotic pairs, and generates focused plots.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path

import pandas as pd

from amr.pipelines.scientific_interest import (
    build_yearly_si_dataset,
    rank_future_high_interest_pairs,
)
from amr.scientific_interest import (
    build_yearly_category_counts,
    plot_yearly_category_driver_counts,
    plot_yearly_si_pair,
    summarize_si_change_drivers,
)


## Configuration


In [ ]:
YEARLY_SCREENED_PATH = Path("yearly_amr_screened_pairs.parquet")
UNIFIED_DECISIONS_PATH = Path("yearly_amr_unified_decisions.parquet")
PATHOGEN_BG_PATH = Path("pathogen_tiab_counts.parquet")
ANTIBIOTIC_BG_PATH = Path("antibiotic_tiab_counts.parquet")

YEARLY_SI_PATH = Path("yearly_si.parquet")
RANK_OUTPUT_PATH = Path("yearly_si_ranked_future.parquet")
INTERESTING_OUTPUT_PATH = Path("yearly_si_ranked_future_interesting.parquet")
DRIVER_OUTPUT_PATH = Path("yearly_si_driver_explanations.parquet")
INTERESTING_PLOT_DIR = Path("interesting_yearly_si_pngs")
DRIVER_PLOT_DIR = Path("interesting_yearly_si_driver_pngs")

FIT_START_YEAR = 2020
FIT_END_YEAR = 2024
NUM_FUTURE_YEARS = 20

MIN_CURRENT_PREDICTED_COUNT = 10
MIN_FUTURE_PREDICTED_COUNT = 10
MIN_SI_CHANGE = 0.1
MIN_POST_FIT_OBSERVED_YEARS = 1
MAX_POST_FIT_SI_ABS_PCT_ERROR_MEAN = 40.0
FALLBACK_TOP_N = 0

CATEGORY_INCLUDE_DECISIONS = ("yes", "uncertain")
DRIVER_LOOKBACK_YEARS = 5
MAX_DRIVER_PLOTS = 25


## Load Canonical Inputs


In [ ]:
yearly_screened_df = pd.read_parquet(YEARLY_SCREENED_PATH)
pathogen_bg_df = pd.read_parquet(PATHOGEN_BG_PATH)
antibiotic_bg_df = pd.read_parquet(ANTIBIOTIC_BG_PATH)

driver_required_cols = {
    "pathogen",
    "antibiotic",
    "year",
    "unified_decision",
    "amr_primary_category",
    "amr_pair_centrality",
    "amr_needs_manual_review",
}
unified_decisions_df = pd.DataFrame()
if UNIFIED_DECISIONS_PATH.exists():
    candidate_unified_df = pd.read_parquet(UNIFIED_DECISIONS_PATH)
    missing_driver_cols = driver_required_cols - set(candidate_unified_df.columns)
    if missing_driver_cols:
        print(
            "Skipping SI driver classification analysis because "
            f"{UNIFIED_DECISIONS_PATH} is missing columns: {sorted(missing_driver_cols)}"
        )
    else:
        unified_decisions_df = candidate_unified_df
else:
    print(
        "Skipping SI driver classification analysis because "
        f"{UNIFIED_DECISIONS_PATH} was not found."
    )

print(f"yearly_screened rows: {len(yearly_screened_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")
print(f"unified decision rows available for driver analysis: {len(unified_decisions_df):,}")


## Build Yearly SI Dataset


In the next code cell, the notebook combines the screened yearly AMR counts with background publication counts to calculate a scientific-interest score for each pathogen-antibiotic-year. In simple terms, it estimates how much attention each pair is getting beyond what would be expected from general research activity alone, then saves those results for ranking and forecasting in later steps.


In [ ]:
si_df, pooled_model, design_columns = build_yearly_si_dataset(
    yearly_screened_df,
    pathogen_bg_df,
    antibiotic_bg_df,
    output_path=YEARLY_SI_PATH,
)

print(f"SI rows: {len(si_df):,}")
print(f"SI pairs: {si_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
si_df.head(10)


## Rank Reliable Future High-Interest Pairs


In [ ]:
rank_df, interesting_df = rank_future_high_interest_pairs(
    si_df,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    min_current_predicted_count=MIN_CURRENT_PREDICTED_COUNT,
    min_future_predicted_count=MIN_FUTURE_PREDICTED_COUNT,
    min_si_change=MIN_SI_CHANGE,
    min_post_fit_observed_years=MIN_POST_FIT_OBSERVED_YEARS,
    max_post_fit_si_abs_pct_error_mean=MAX_POST_FIT_SI_ABS_PCT_ERROR_MEAN,
    fallback_top_n=FALLBACK_TOP_N,
    rank_output_path=RANK_OUTPUT_PATH,
    interesting_output_path=INTERESTING_OUTPUT_PATH,
    verbose=True,
)

interesting_df[
    [
        "pathogen",
        "antibiotic",
        "interest_score",
        "si_change",
        "current_predicted_count",
        "future_predicted_count",
        "post_fit_si_abs_pct_error_mean",
    ]
].head(30)


## Explain Predicted SI Changes

This section uses the LLM category metadata from the unified abstract screen to describe which AMR paper types are driving the observed count curves behind the SI forecast. The analysis is descriptive: it explains category mix and recent changes, while the ranking itself remains based on the screened yearly AMR counts.


In [ ]:
category_counts_df = build_yearly_category_counts(
    unified_decisions_df,
    include_decisions=CATEGORY_INCLUDE_DECISIONS,
)
driver_summary_df = summarize_si_change_drivers(
    interesting_df,
    unified_decisions_df,
    fit_end_year=FIT_END_YEAR,
    include_decisions=CATEGORY_INCLUDE_DECISIONS,
    lookback_years=DRIVER_LOOKBACK_YEARS,
)

if driver_summary_df.empty:
    interesting_with_drivers_df = interesting_df.copy()
    print("No SI driver classification summaries were produced.")
else:
    interesting_with_drivers_df = interesting_df.merge(
        driver_summary_df,
        on=["pathogen", "antibiotic"],
        how="left",
    )
    interesting_with_drivers_df.to_parquet(DRIVER_OUTPUT_PATH, index=False)
    print(f"Saved SI driver explanations to {DRIVER_OUTPUT_PATH}")

driver_display_cols = [
    "pathogen",
    "antibiotic",
    "interest_score",
    "si_change",
    "current_predicted_count",
    "future_predicted_count",
    "driver_recent_total",
    "driver_count_change",
    "driver_dominant_category",
    "driver_fastest_growing_category",
    "driver_weak_signal_share",
    "driver_manual_review_share",
]
available_driver_cols = [col for col in driver_display_cols if col in interesting_with_drivers_df.columns]
interesting_with_drivers_df[available_driver_cols].head(30)


## Plot Category Driver Curves


In [ ]:
DRIVER_PLOT_DIR.mkdir(parents=True, exist_ok=True)
saved_driver_plots = 0

if category_counts_df.empty:
    print("No category count curves available to plot.")
else:
    for row in interesting_df.head(MAX_DRIVER_PLOTS).itertuples(index=False):
        try:
            result = plot_yearly_category_driver_counts(
                category_counts_df=category_counts_df,
                pathogen=row.pathogen,
                antibiotic=row.antibiotic,
                fit_end_year=FIT_END_YEAR,
            )
        except ValueError:
            continue
        pathogen_slug = str(row.pathogen).replace("/", "_").replace(" ", "_")
        antibiotic_slug = str(row.antibiotic).replace("/", "_").replace(" ", "_")
        out_path = DRIVER_PLOT_DIR / f"{pathogen_slug}__{antibiotic_slug}__drivers.png"
        result["figure"].savefig(out_path, dpi=200, bbox_inches="tight")
        saved_driver_plots += 1

    print(f"Saved {saved_driver_plots} category driver plots to {DRIVER_PLOT_DIR}")


## Plot Top Interesting Pairs


In [ ]:
INTERESTING_PLOT_DIR.mkdir(parents=True, exist_ok=True)

for row in interesting_df.head(25).itertuples(index=False):
    result = plot_yearly_si_pair(
        si_df=si_df,
        pathogen=row.pathogen,
        antibiotic=row.antibiotic,
        fit_start_year=FIT_START_YEAR,
        fit_end_year=FIT_END_YEAR,
        num_additional_years=NUM_FUTURE_YEARS,
        loess_frac=0.6,
        loess_degree=2,
        tail_damping_years=None,
    )
    pathogen_slug = str(row.pathogen).replace("/", "_").replace(" ", "_")
    antibiotic_slug = str(row.antibiotic).replace("/", "_").replace(" ", "_")
    out_path = INTERESTING_PLOT_DIR / f"{pathogen_slug}__{antibiotic_slug}.png"
    result["figure"].savefig(out_path, dpi=200, bbox_inches="tight")

print(f"Saved {min(25, len(interesting_df))} plots to {INTERESTING_PLOT_DIR}")
